# T10 — Flight Option Finder Agent

A simple live demonstration of the Agentic AI project.

**Goal:** Find and pick a flight within the user's budget.

This notebook demonstrates:
- Live flight search using SerpApi
- Price comparison
- Multi-step agent behavior
- Memory across turns
- Handling a budget with no matching flight


In [6]:
# ============================================================
# SETUP
# ============================================================

import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# ------------------------------------------------------------
# Find the project directory
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

# If notebook is not running from the project folder,
# search upward for the folder containing app/
while PROJECT_ROOT != PROJECT_ROOT.parent:

    if (PROJECT_ROOT / "app").is_dir():
        break

    PROJECT_ROOT = PROJECT_ROOT.parent

# ------------------------------------------------------------
# If app/ still isn't found, use the known project location
# ------------------------------------------------------------

if not (PROJECT_ROOT / "app").is_dir():

    possible_project = Path(
        r"C:\Users\Ayush Yadav\Downloads\CSE476CA1\flight_option_finder"
    )

    if (possible_project / "app").is_dir():
        PROJECT_ROOT = possible_project

# ------------------------------------------------------------
# Add project to Python path
# ------------------------------------------------------------

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ------------------------------------------------------------
# Load .env from the project directory
# ------------------------------------------------------------

load_dotenv(
    PROJECT_ROOT / ".env",
    override=True
)

# ------------------------------------------------------------
# Verify project
# ------------------------------------------------------------

print("Project root:")
print(PROJECT_ROOT)

print("\napp folder found:")
print((PROJECT_ROOT / "app").is_dir())

print("\nSerpApi configured:")
print(bool(os.getenv("SERPAPI_KEY")))

print("\nGroq configured:")
print(bool(os.getenv("GROQ_API_KEY")))

# ------------------------------------------------------------
# Import agent
# ------------------------------------------------------------

from app.agent import FlightAgent, build_client

agent = FlightAgent(
    client=build_client()
)

print("\nAgent ready!")

Project root:
C:\Users\Ayush Yadav\Downloads\CSE476CA1\flight_option_finder

app folder found:
True

SerpApi configured:
True

Groq configured:
True

Agent ready!


## Live Demo 1 — Find a Flight

**User request:**

> Find me a flight from Delhi to Mumbai on September 15, 2026. My budget is ₹7,000. I am traveling alone in economy class. Prefer a morning flight.

The agent should:
1. Search live flights.
2. Compare the prices.
3. Apply the ₹7,000 budget.
4. Prefer a morning flight.
5. Return the best suitable option.


In [7]:
# 2. Live Demo 1

result1 = agent.run(
    """
    Find me a flight from Delhi to Mumbai
    on September 15, 2026.

    My budget is ₹7000.

    I am traveling alone in economy class.

    Prefer a morning flight.
    """
)

print("FINAL ANSWER")
print("=" * 70)
print(result1["answer"])

print("\nAGENT TRACE")
print("=" * 70)

for item in result1["trace"]:
    print(item)


FINAL ANSWER
**Best option for you**

| Airline | Flight | Departure (DEL) → Arrival (BOM) | Price | Cabin | Stops |
|---------|--------|----------------------------------|-------|-------|-------|
| **Air India** | AI ‑ 2026‑09‑15‑05:00 | 05:00 – 07:20 (≈ 2 h 20 m) | **₹6,314** | Economy | Non‑stop |

**Why this flight fits your needs**

- **Budget:** ₹6,314 < ₹7,000 (well within your limit).  
- **Morning preference:** Departs at 05:00 am, clearly a morning flight.  
- **Convenience:** Non‑stop, short 2 h 20 m journey, economy class, solo traveler.

If you’d like to book this flight or need any additional details (e.g., baggage allowance, seat selection, or alternative morning options), just let me know!

AGENT TRACE
[state] Budget=7000; Preferences=['morning']; Route=('Delhi', 'Mumbai'); Departure date=2026-09-15 | departure_date=2026-09-15
[step 1] search_flights({'cabin_class': 'economy', 'departure_date': '2026-09-15', 'from_city': 'Delhi', 'passengers': 1, 'to_city': 'Mumbai'})
[

## Live Demo 2 — Memory

Now we ask a follow-up question **without repeating the original requirements**.

The same agent instance is used, so it can use the previous conversation and stored memory.


In [9]:
# 3. Live Demo 2 — Memory

result2 = agent.run(
    "What is the best option from my saved search?"
)

print("FINAL ANSWER")
print("=" * 70)
print(result2["answer"])

print("\nMEMORY")
print("=" * 70)
print(result2["memory"])


FINAL ANSWER
**Best option for you**

| Airline | Flight | Departure (DEL) → Arrival (BOM) | Price | Cabin | Stops |
|---------|--------|----------------------------------|-------|-------|-------|
| **Air India** | AI ‑ 2026‑09‑15‑05:00 | 05:00 – 07:20 (≈ 2 h 20 m) | **₹6,314** | Economy | Non‑stop |

**Why this flight fits your needs**

- **Budget:** ₹6,314 < ₹7,000 (well within your limit).  
- **Morning preference:** Departs at 05:00 am, clearly a morning flight.  
- **Convenience:** Non‑stop, short 2 h 20 m journey, economy class, solo traveler.

If you’d like to book this flight or need any additional details (e.g., baggage allowance, seat selection, or alternative morning options), just let me know!

MEMORY
{'messages': [{'role': 'user', 'content': '\n    Find me a flight from Delhi to Mumbai\n    on September 15, 2026.\n\n    My budget is ₹7000.\n\n    I am traveling alone in economy class.\n\n    Prefer a morning flight.\n    '}, {'role': 'assistant', 'content': '**Best option 

## Live Demo 3 — No Flight Within Budget

This test uses a deliberately low budget.

The agent should **not invent a flight**. It should report that no available flight fits the budget and identify the cheapest available option.


In [5]:
# 4. Live Demo 3 — No Fit

result3 = agent.run(
    """
    Find me a flight from Delhi to Bengaluru
    on September 15, 2026.

    My budget is ₹4000.

    I am traveling alone in economy class.
    """
)

print("FINAL ANSWER")
print("=" * 70)
print(result3["answer"])

print("\nAGENT TRACE")
print("=" * 70)

for item in result3["trace"]:
    print(item)


FINAL ANSWER
**Result:** No flight on 2026‑09‑15 from Delhi (DEL) to Bengaluru (BLR) in economy class is available within your budget of **₹ 4,000**.

**Cheapest option found:**  

- **Airline:** IndiGo  
- **Departure:** 2026‑09‑15 10:45 (Delhi) → 13:40 (Bengaluru)  
- **Price:** **₹ 8,724** (non‑stop, 175 min)  

All other flights listed are priced at ₹ 8,724 or higher, which exceeds your budget. If you can increase your budget, the IndiGo flight at 10:45 am (or the similar 12:20 pm option) would be the most affordable choice. Let me know if you’d like to explore alternatives such as a different date, a higher budget, or a different cabin class.

AGENT TRACE
[state] Budget=4000; Preferences=['morning']; Route=('Delhi', 'Bengaluru'); Departure date=2026-09-15 | departure_date=2026-09-15
[step 1] search_flights({'cabin_class': 'economy', 'departure_date': '2026-09-15', 'from_city': 'Delhi', 'passengers': 1, 'to_city': 'Bengaluru'})
[OK] -> Flight search results: DEL → BLR
Departure dat

## Result

The three demonstrations show the required T10 agent behavior:

- **Tool use:** flight search and price comparison
- **Multi-step reasoning:** search → compare → decide
- **Memory:** previous requirements are available in a follow-up turn
- **Budget decision:** suitable flight is selected when possible
- **Failure handling:** the agent reports when no flight fits

> **Note:** Flight prices and availability are live and may change between runs.
